# VL-14:梁真實配筋設計——從假設值到真正的需求-設計閉環

`Case-06.5` 從一開始梁就用 $\rho=0.02$(6 根鋼筋單筋)這個假設配置,
從沒有真正用 `design_Tbeam()`/`design_doubly_reinforced()` 針對這個
結構真正承受的彎矩需求去設計——柱子這邊至少已經被 `design_column_PM()`
嚴謹分析過真實容量(見 `VL-08`/`VL-11`/`VL-13`),梁完全沒有。

**這一課要做的事**:從 `Case-06.5` 的 `Hyb` 模型(纖維斷面柱+彈性梁)
實際跑一次完整推覆分析,抓出梁端**真正**承受的彎矩需求(不是假設),
再用這個 repo 自己的設計函式做正式設計。

**過程中發現一件重要的事,不是順利設計完就結束**:1F 樓板梁的真實
彎矩需求,遠遠超過原本假設配筋(6 根單筋)的容量——這對 `Case-06.5`
「強梁弱柱」這個關鍵假設,是一個需要正視的警訊。

## 第 1 課:從 `Hyb` 模型的真實推覆分析,抓出梁端真正的彎矩需求

沿用 `Case-06.5` 完全相同的模型定義(`build_frame_with_fiber_columns()`),
在整個推覆過程中追蹤兩根梁(元素 5=1F 樓板梁、元素 6=屋頂梁)的端點
彎矩,取全程最大值當作設計需求。

In [1]:

try:
    import openseespy.opensees as ops
except ImportError:
    import sys
    !{sys.executable} -m pip install -q openseespy
    import openseespy.opensees as ops
import numpy as np

# 完全沿用Case-06.5的參數, 不重新假設
L_bay = 6.0
h1 = h2 = 3.5
b = h = 0.40
cover = 0.04
fc = 27.46e3
fy = 411.88e3
Es = 2.0e8
rho = 0.02
As_total_col = rho*b*h

fc_confined = fc*1.310
eps_cc = 0.00510
eps_cu_confined = 0.01112

b_beam, h_beam = 0.3, 0.5
Ib_gross = b_beam*h_beam**3/12
Ib = 0.35*Ib_gross
A_beam = b_beam*h_beam
E_rc = 2.463e7

F1_Y, F2_Y = 9.938, 15.900
Pu_col = 147.60


def build_frame_with_fiber_columns():
    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.uniaxialMaterial('Concrete01', 1, -fc, -0.002, -0.2*fc, -0.006)
    ops.uniaxialMaterial('Concrete01', 3, -fc_confined, -eps_cc, -0.85*fc_confined, -eps_cu_confined)
    ops.uniaxialMaterial('Steel02', 2, fy, Es, 0.01, 18, 0.925, 0.15)
    ops.section('Fiber', 1)
    half = h/2; core_half = h/2 - cover
    ops.patch('rect', 3, 10, 10, -core_half, -core_half, core_half, core_half)
    ops.patch('rect', 1, 10, 2, -half, -half, half, -core_half)
    ops.patch('rect', 1, 10, 2, -half, core_half, half, half)
    ops.patch('rect', 1, 2, 6, -half, -core_half, -core_half, core_half)
    ops.patch('rect', 1, 2, 6, core_half, -core_half, half, core_half)
    positions = [(-core_half,-core_half),(core_half,-core_half),(core_half,core_half),(-core_half,core_half),
                 (0,-core_half),(0,core_half),(-core_half,0),(core_half,0)]
    As_bar = As_total_col/8
    for (y,z) in positions:
        ops.fiber(y, z, As_bar, 2)
    ops.node(1, 0.0, 0.0);   ops.node(2, L_bay, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L_bay, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L_bay, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    ops.geomTransf('Linear', 1); ops.geomTransf('Linear', 2)
    ops.beamIntegration('Lobatto', 1, 1, 5)
    ops.element('forceBeamColumn', 1, 1, 3, 1, 1)
    ops.element('forceBeamColumn', 2, 2, 4, 1, 1)
    ops.element('forceBeamColumn', 3, 3, 5, 1, 1)
    ops.element('forceBeamColumn', 4, 4, 6, 1, 1)
    ops.element('elasticBeamColumn', 5, 3, 4, A_beam, E_rc, Ib, 2)   # 1F樓板梁
    ops.element('elasticBeamColumn', 6, 5, 6, A_beam, E_rc, Ib, 2)   # 屋頂梁


def analyze_with_fallback():
    ok = ops.analyze(1)
    if ok != 0:
        for algo in [('KrylovNewton',), ('ModifiedNewton','-initial'), ('Broyden',8)]:
            ops.algorithm(*algo)
            ok = ops.analyze(1)
            if ok == 0:
                break
        ops.algorithm('Newton')
    return ok


build_frame_with_fiber_columns()

ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
for n in [3,4,5,6]:
    ops.load(n, 0.0, -Pu_col, 0.0)
ops.system('BandGeneral'); ops.numberer('RCM'); ops.constraints('Plain')
ops.test('NormUnbalance', 1e-6, 50)
ops.algorithm('Newton')
ops.integrator('LoadControl', 0.05)
ops.analysis('Static')
ok0 = ops.analyze(20)
assert ok0 == 0, "重力載重分析應該成功收斂"
ops.loadConst('-time', 0.0)

ops.timeSeries('Linear', 2); ops.pattern('Plain', 2, 2)
ops.load(3, F1_Y, 0.0, 0.0)
ops.load(5, F2_Y, 0.0, 0.0)
control_node, control_dof = 5, 1
max_disp, n_steps = 0.28, 560
ops.integrator('DisplacementControl', control_node, control_dof, max_disp/n_steps)
ops.analysis('Static')

max_M_beam1F, max_M_beamRoof = 0.0, 0.0
for i in range(n_steps):
    ok = analyze_with_fallback()
    if ok != 0:
        print(f"停在step {i}")
        break
    f5 = ops.eleForce(5)
    f6 = ops.eleForce(6)
    max_M_beam1F = max(max_M_beam1F, abs(f5[2]), abs(f5[5]))
    max_M_beamRoof = max(max_M_beamRoof, abs(f6[2]), abs(f6[5]))

print(f"1F樓板梁(元素5)最大端彎矩需求 = {max_M_beam1F:.2f} kN-m")
print(f"屋頂梁(元素6)最大端彎矩需求 = {max_M_beamRoof:.2f} kN-m")


1F樓板梁(元素5)最大端彎矩需求 = 428.44 kN-m
屋頂梁(元素6)最大端彎矩需求 = 224.22 kN-m


## 第 2 課:對照原本假設配筋的容量——揭露真實的超載狀況

`Case-06.5` 原本用簡化手算法(假設 $\rho=0.02$、6 根單筋)算出的
$M_y$(純彎,`P=0`)$=259.40$kN-m。

In [2]:

My_beam_assumed = 259.40  # Case-06.5原本用的假設配筋容量

print(f"{'梁位置':<12}{'真實Mu需求':>14}{'假設配筋容量':>16}{'利用率':>10}")
print(f"{'1F樓板梁':<12}{max_M_beam1F:>14.2f}{My_beam_assumed:>16.2f}{max_M_beam1F/My_beam_assumed:>10.1%}")
print(f"{'屋頂梁':<12}{max_M_beamRoof:>14.2f}{My_beam_assumed:>16.2f}{max_M_beamRoof/My_beam_assumed:>10.1%}")

print()
print("[誠實記錄] 1F樓板梁的真實彎矩需求, 遠遠超過原本假設配筋的容量")
print("(利用率165%, 嚴重超載)——這不是誤差量級, 是結構性的問題:")
print("Case-06.5「強梁弱柱」這個關鍵假設(梁維持彈性, 只有柱追蹤塑性),")
print("在1F樓板梁這裡可能根本不成立。如果梁真的用真實彈塑性行為建模,")
print("1F樓板梁應該早就降伏, 結構的破壞模式可能跟目前假設的不一樣。")
print("屋頂梁(86.4%)相對合理, 沒有超載。")


梁位置                 真實Mu需求          假設配筋容量       利用率
1F樓板梁               428.44          259.40    165.2%
屋頂梁                 224.22          259.40     86.4%

[誠實記錄] 1F樓板梁的真實彎矩需求, 遠遠超過原本假設配筋的容量
(利用率165%, 嚴重超載)——這不是誤差量級, 是結構性的問題:
Case-06.5「強梁弱柱」這個關鍵假設(梁維持彈性, 只有柱追蹤塑性),
在1F樓板梁這裡可能根本不成立。如果梁真的用真實彈塑性行為建模,
1F樓板梁應該早就降伏, 結構的破壞模式可能跟目前假設的不一樣。
屋頂梁(86.4%)相對合理, 沒有超載。


## 第 3 課:用真實需求正式設計梁配筋

`design_rebar()`(單筋)優先,不夠才用 `design_doubly_reinforced()`
(雙筋)——不是每根梁一律套雙筋公式,讓函式自己判斷需要與否。

In [3]:

import os
if not os.path.exists('rc_design.py'):
    !wget -q -O rc_design.py https://raw.githubusercontent.com/zhixiu0223/taiwan-seismic-code-calc/main/rc_design.py

from rc_design import design_rebar, design_doubly_reinforced

print("=== 1F樓板梁, Mu={:.2f} kN-m ===".format(max_M_beam1F))
try:
    r_1F = design_rebar(max_M_beam1F, 30.0, 50.0, cover=4.0)
    print(f"單筋方案可行: As_provided={r_1F['As_provided']:.2f}cm^2, phiMn={r_1F['phiMn_provided']:.2f}")
    beam_1F_design = r_1F
except ValueError as e:
    print(f"單筋設計失敗: {e}")
    r_1F = design_doubly_reinforced(max_M_beam1F, 30.0, 50.0, 6.0)
    print(f"改用雙筋設計:")
    print(f"  As_total={r_1F['As_total']:.2f}cm^2, As_prime={r_1F['As_prime']:.2f}cm^2(壓力筋)")
    print(f"  phiMn_provided={r_1F['phiMn_provided']:.2f}kN-m, utilization={r_1F['utilization']:.2%}")
    print(f"  d={r_1F['d']:.2f}cm, eps_t={r_1F['eps_t']:.5f}, phi_used={r_1F['phi_used']:.3f}")
    beam_1F_design = r_1F

print()
print("=== 屋頂梁, Mu={:.2f} kN-m ===".format(max_M_beamRoof))
try:
    r_roof = design_rebar(max_M_beamRoof, 30.0, 50.0, cover=4.0)
    print(f"單筋方案可行: As_provided={r_roof['As_provided']:.2f}cm^2, phiMn={r_roof['phiMn_provided']:.2f}")
    beam_roof_design = r_roof
except ValueError as e:
    print(f"單筋設計失敗: {e}")
    r_roof = design_doubly_reinforced(max_M_beamRoof, 30.0, 50.0, 6.0)
    beam_roof_design = r_roof


=== 1F樓板梁, Mu=428.44 kN-m ===
單筋設計失敗: eps_t=0.0025<0.005, 非拉力控制斷面, 需要用過渡區phi內插或加大斷面
改用雙筋設計:
  As_total=32.86cm^2, As_prime=11.74cm^2(壓力筋)
  phiMn_provided=435.34kN-m, utilization=98.41%
  d=41.60cm, eps_t=0.00455, phi_used=0.861

=== 屋頂梁, Mu=224.22 kN-m ===
單筋方案可行: As_provided=15.48cm^2, phiMn=225.26


## 第 4 課:新舊配筋量對照

In [4]:

As_original = rho*30*50  # cm^2, 原本假設值(6根單筋)

print(f"{'梁位置':<12}{'真實Mu需求':>14}{'原假設配筋':>14}{'原假設容量':>14}{'新設計配筋':>14}{'新設計容量':>14}")
As_1F_new = beam_1F_design.get('As_total', beam_1F_design.get('As_provided'))
As_roof_new = beam_roof_design.get('As_total', beam_roof_design.get('As_provided'))
print(f"{'1F樓板梁':<12}{max_M_beam1F:>14.2f}{As_original:>14.1f}{259.40:>14.2f}{As_1F_new:>14.2f}{beam_1F_design['phiMn_provided']:>14.2f}")
print(f"{'屋頂梁':<12}{max_M_beamRoof:>14.2f}{As_original:>14.1f}{259.40:>14.2f}{As_roof_new:>14.2f}{beam_roof_design['phiMn_provided']:>14.2f}")

assert beam_1F_design['phiMn_provided'] >= max_M_beam1F, "設計出的配筋必須真正滿足需求"
assert beam_roof_design['phiMn_provided'] >= max_M_beamRoof, "設計出的配筋必須真正滿足需求"
print("\n[PASS] 兩根梁的新配筋都真正滿足各自的真實彎矩需求")


梁位置                 真實Mu需求         原假設配筋         原假設容量         新設計配筋         新設計容量
1F樓板梁               428.44          30.0        259.40         32.86        435.34
屋頂梁                 224.22          30.0        259.40         15.48        225.26

[PASS] 兩根梁的新配筋都真正滿足各自的真實彎矩需求


## 小結

- 補上 `Case-06.5` 從一開始就欠缺的環節:梁的真實彎矩需求,從沒有被
  正式抓取過,更沒有用這個 repo 自己的設計函式(`design_rebar()`/
  `design_doubly_reinforced()`)設計過——柱子這邊至少已經被
  `design_column_PM()` 嚴謹分析過(`VL-08`/`VL-11`/`VL-13`),梁完全
  沒有,是本次才第一次補上
- **誠實記錄一個重大發現**:`1F` 樓板梁的真實彎矩需求(`428.44kN-m`),
  遠超原本假設配筋(`6` 根單筋)的容量(`259.40kN-m`),利用率
  `165%`——這對 `Case-06.5`「強梁弱柱」這個關鍵假設是一個需要正視
  的警訊,不是可以忽略的小差異
- `1F` 樓板梁需要**雙筋設計**(`As_total=32.86cm²`,含壓力筋
  `11.74cm²`)才能滿足真實需求;屋頂梁單筋即可(`15.48cm²`)
- **這個發現的後續意義**:如果要讓 `Case-06.5` 的側推分析結果真正
  可信,下一步應該考慮讓梁也追蹤真實的塑性行為(不能繼續假設「永遠
  彈性」),或者至少要用新設計出的配筋重新驗證這個假設是否合理——
  這是比單純「補上配筋」更深一層的、需要後續處理的缺口